# Static EEG Data Wrangling

This notebook builds and analyzes a combined static dataset from CHB-MIT and Siena scalp EEG datasets

## Notebook goals

- Configure dataset and preprocessing settings.
- Run the static dataset-building pipeline script.
- Load the generated dataset outputs for inspection and EDA.


## Running Instructions

This notebook works in both **VS Code** and **Google Colab**.

### Local (VS Code)
- Run all cells from top to bottom.

### Google Colab
- Upload one of the following files to Google Colab:
  - `master_eeg_dataset.parquet`
  - `master_eeg_dataset.csv`
- Then **skip to Step 2: Load the Dataset**

## Requirements and Outputs

- `DATASET_ROOT_OVERRIDE` from the configuration cell below.

The dataset root must contain the folders `chbmit/` and `siena/`.

Outputs from this notebook:

- `master_eeg_dataset.parquet`
- `master_eeg_dataset.csv` 
- `master_eeg_dataset_X.npy` 
- `master_eeg_dataset_y.npy`
- `master_eeg_dataset_meta.csv`


## Imports

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import sys
import os

from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.metrics import precision_score, recall_score, f1_score

## Configuration

This section keeps the settings most people are likely to tweak.

Inputs: Dataset root override, set preprocessing and epoch parameters.

Output for the next step: notebook configuration variables used by the bootstrap and workflow cells.


In [ ]:
# Explicit dataset root. Change this to where you store the datasets
DATASET_ROOT_OVERRIDE = None

# Static dataset configuration
EPOCH_DURATION_SEC = 10.0
EPOCH_OVERLAP_SEC = 0.0
TARGET_SFREQ = 256
PREICTAL_HORIZON_SEC = 600
POSTICTAL_EXCLUSION_SEC = 1800
INTERICTAL_GAP_SEC = 300

## Step 1: Build the Dataframe

This step puts both static datasets into a dataframe, by running the pipeline script named build_static_dataset

In [3]:
repo_root = Path.cwd()
output_root = repo_root / "artifacts" / "static"
output_root.mkdir(parents=True, exist_ok=True)

dataset_root_str = DATASET_ROOT_OVERRIDE

cmd = [
    sys.executable,
    "-m",
    "scripts.build_static_dataset",
    "--dataset-root",
    dataset_root_str,
    "--output-root",
    str(output_root),
    "--epoch-duration",
    str(EPOCH_DURATION_SEC),
    "--epoch-overlap",
    str(EPOCH_OVERLAP_SEC),
    "--preictal-horizon-sec",
    str(PREICTAL_HORIZON_SEC),
    "--postictal-exclusion-sec",
    str(POSTICTAL_EXCLUSION_SEC),
    "--interictal-gap-sec",
    str(INTERICTAL_GAP_SEC),
]

print("Running command:")
print(" ".join(cmd))

env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root / "src")

result = subprocess.run(
    cmd,
    cwd=repo_root,
    env=env,
    text=True,
    capture_output=True,
)

print("STDOUT:\n")
print(result.stdout)

if result.stderr:
    print("STDERR:\n")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"Dataset build failed with exit code {result.returncode}")

print("\nBuild completed.")
print(f"Outputs saved in: {output_root}")
print("Files:")
for p in sorted(output_root.iterdir()):
    print(" -", p.name)

Running command:
c:\Users\arnol\AppData\Local\Programs\Python\Python313\python.exe -m scripts.build_static_dataset --dataset-root D:/ --output-root c:\Developer\Python\EEG Project\eeg-Spr2026-CSCI7090\artifacts\static --epoch-duration 10.0 --epoch-overlap 0.0 --preictal-horizon-sec 600 --postictal-exclusion-sec 1800 --interictal-gap-sec 300
STDOUT:

[DEBUG] About to load annotations
[DEBUG] Reading Siena annotation file: D:\siena\LICENSE.txt
[DEBUG] Reading Siena annotation file: D:\siena\SHA256SUMS.txt
[DEBUG] Siena annotation file had possible content but no parsed intervals: D:\siena\SHA256SUMS.txt
[DEBUG] Reading Siena annotation file: D:\siena\PN00\Seizures-list-PN00.txt
[DEBUG] Parsed 5 Siena seizure intervals from D:\siena\PN00\Seizures-list-PN00.txt
[DEBUG] Reading Siena annotation file: D:\siena\PN01\Seizures-list-PN01.txt
[DEBUG] Siena annotation file had possible content but no parsed intervals: D:\siena\PN01\Seizures-list-PN01.txt
[DEBUG] Reading Siena annotation file: D:\s

## Step 2: Load the Dataset

This cell loads the preprocessed EEG dataset from either CSV or parquet file into a DataFrame. This step also checks for missing and duplicate values.

In [ ]:
possible_parquet_paths = [
    Path("artifacts/static/master_eeg_dataset.parquet"),
    Path("master_eeg_dataset.parquet"),
]

possible_csv_paths = [
    Path("artifacts/static/master_eeg_dataset.csv"),
    Path("master_eeg_dataset.csv"),
]

parquet_path = next((p for p in possible_parquet_paths if p.exists()), None)
csv_path = next((p for p in possible_csv_paths if p.exists()), None)

if parquet_path is not None:
    print("Loading dataset from:", parquet_path)
    df_all = pd.read_parquet(parquet_path)

elif csv_path is not None:
    print("Loading dataset from:", csv_path)
    try:
        df_all = pd.read_csv(csv_path)
    except Exception:
        df_all = pd.read_csv(csv_path, engine="python", on_bad_lines="skip")

else:
    raise FileNotFoundError(
        "Could not find master_eeg_dataset.parquet or master_eeg_dataset.csv.\n"
        "Place the dataset in 'artifacts/static/' or upload it directly to the working directory."
    )

print("Shape:", df_all.shape)
print("Columns:", df_all.columns.tolist())
print("Number of missing values:", df_all.isna().sum().sum())
print("Number of duplicate rows:", df_all.duplicated().sum())
display(df_all.groupby("dataset").head(5))

Loading dataset from: artifacts\static\master_eeg_dataset.parquet
Shape: (354478, 72)
Columns: ['source_type', 'dataset', 'subject_id', 'session_id', 'record_id', 'edf_path', 'source_file_name', 'epoch_index', 'window_start_sec', 'window_end_sec', 'duration_sec', 'label', 'target', 'n_seizures_in_file', 'feature_version', 'channel_schema', 'channel_signature', 'annotation_source', 'mean', 'std', 'min', 'max', 'range', 'energy', 'rms', 'abs_mean', 'line_length', 'zero_crossing_rate', 'spectral_entropy', 'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity', 'delta_power', 'theta_power', 'alpha_power', 'beta_power', 'gamma_power', 'total_power', 'delta_relative_power', 'theta_relative_power', 'alpha_relative_power', 'beta_relative_power', 'gamma_relative_power', 'dominant_frequency', 'spectral_centroid', 'spectral_bandwidth', 'theta_beta_ratio', 'delta_alpha_ratio', 'alpha_beta_ratio', 'delta_theta_ratio', 'theta_alpha_ratio', 'delta_beta_ratio', 'low_high_ratio', 'line_length_channe

,source_type,dataset,subject_id,session_id,record_id,edf_path,source_file_name,epoch_index,window_start_sec,window_end_sec,...,theta_power_channel_std,alpha_power_channel_mean,alpha_power_channel_std,beta_power_channel_mean,beta_power_channel_std,gamma_power_channel_mean,gamma_power_channel_std,n_channels,n_samples,sfreq
0,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000000,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,0,0.0,10.0,...,0.000009,0.000003,0.000002,0.000002,0.000002,2.162419e-06,2.172604e-06,22,2560,256.0
1,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000001,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,1,10.0,20.0,...,0.000014,0.000003,0.000002,0.000004,0.000005,4.061062e-06,5.093390e-06,22,2560,256.0
2,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000002,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,2,20.0,30.0,...,0.000014,0.000003,0.000003,0.000002,0.000002,2.223560e-06,2.779754e-06,22,2560,256.0
3,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000003,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,3,30.0,40.0,...,0.000008,0.000002,0.000001,0.000002,0.000002,1.918444e-06,1.957092e-06,22,2560,256.0
4,static,chbmit,chb01,chb01_01,chbmit:chb01:chb01_01:epoch_000004,D:\chbmit\chb01\chb01_01.edf,chb01_01.edf,4,40.0,50.0,...,0.000012,0.000003,0.000001,0.000001,0.000001,9.331016e-07,9.178800e-07,22,2560,256.0
329893,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000000,D:\siena\PN00\PN00-1.edf,PN00-1.edf,0,0.0,10.0,...,0.000592,0.000077,0.000409,0.000032,0.000165,4.326854e-06,1.707255e-05,35,2560,256.0
329894,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000182,D:\siena\PN00\PN00-1.edf,PN00-1.edf,182,1820.0,1830.0,...,0.000392,0.000058,0.000298,0.000025,0.000125,2.019241e-06,9.632130e-06,35,2560,256.0
329895,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000183,D:\siena\PN00\PN00-1.edf,PN00-1.edf,183,1830.0,1840.0,...,0.000466,0.000051,0.000262,0.000023,0.000114,1.938598e-06,9.034187e-06,35,2560,256.0
329896,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000184,D:\siena\PN00\PN00-1.edf,PN00-1.edf,184,1840.0,1850.0,...,0.000482,0.000057,0.000292,0.000025,0.000124,1.927358e-06,9.500415e-06,35,2560,256.0
329897,static,siena,pn00,pn00-1,siena:pn00:pn00-1:epoch_000185,D:\siena\PN00\PN00-1.edf,PN00-1.edf,185,1850.0,1860.0,...,0.000526,0.000055,0.000281,0.000025,0.000122,1.971159e-06,9.666390e-06,35,2560,256.0


## Step 3: Visualizations

This step calls the visualization script

In [37]:
!python ../../scripts/eeg_eda_visualizations.py

## Step 4: Data Preparation and Train/Test Split
This step selects time domain features and identifies the target label. It also splits the data based on subject_id, which avoids data leakage and produces more realistic model results. Additionally, the large class imbalance between preictal and interictal samples is addressed by calculating class weights.

In [ ]:
target_col = "target"
subject_col = "subject_id"

feature_cols = [
    # time-domain
    "mean",
    "std",
    "min",
    "max",
    "range",
    "energy",
    "rms",
    "abs_mean",
    "line_length",
    "zero_crossing_rate",
    # complexity
    "spectral_entropy",
    "hjorth_activity",
    "hjorth_mobility",
    "hjorth_complexity",
    # frequency
    "delta_power",
    "theta_power",
    "alpha_power",
    "beta_power",
    "gamma_power",
    "total_power",
    # relative power
    "delta_relative_power",
    "theta_relative_power",
    "alpha_relative_power",
    "beta_relative_power",
    "gamma_relative_power",
    # spectral shape
    "dominant_frequency",
    "spectral_centroid",
    "spectral_bandwidth",
    # ratios
    "theta_beta_ratio",
    "delta_alpha_ratio",
    "alpha_beta_ratio",
    "delta_theta_ratio",
    "theta_alpha_ratio",
    "delta_beta_ratio",
    "low_high_ratio",
    # channel-aware time-domain
    "line_length_channel_mean",
    "line_length_channel_std",
    "rms_channel_mean",
    "rms_channel_std",
    "std_channel_mean",
    "std_channel_std",
    # channel-aware frequency
    "delta_power_channel_mean",
    "delta_power_channel_std",
    "theta_power_channel_mean",
    "theta_power_channel_std",
    "alpha_power_channel_mean",
    "alpha_power_channel_std",
    "beta_power_channel_mean",
    "beta_power_channel_std",
    "gamma_power_channel_mean",
    "gamma_power_channel_std",
]

# feature and target label
X = df_all[feature_cols].copy()
y = df_all[target_col].copy()
groups = df_all[subject_col].copy()

print("Using features:", feature_cols)
print("Unique subjects:", groups.nunique())

# This section splits data into train/test
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)

# splits rows into train/test based on subject_id, this prevents data leakage
train_rows, test_rows = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_rows]
X_test = X.iloc[test_rows]

y_train = y.iloc[train_rows]
y_test = y.iloc[test_rows]

groups_train = groups.iloc[train_rows]
groups_test = groups.iloc[test_rows]

print("Shared subjects:", len(set(groups_train).intersection(set(groups_test))))

Using features: ['mean', 'std', 'min', 'max', 'range', 'energy', 'rms', 'abs_mean', 'line_length', 'zero_crossing_rate', 'spectral_entropy', 'hjorth_activity', 'hjorth_mobility', 'hjorth_complexity', 'delta_power', 'theta_power', 'alpha_power', 'beta_power', 'gamma_power', 'total_power', 'delta_relative_power', 'theta_relative_power', 'alpha_relative_power', 'beta_relative_power', 'gamma_relative_power', 'dominant_frequency', 'spectral_centroid', 'spectral_bandwidth', 'theta_beta_ratio', 'delta_alpha_ratio', 'alpha_beta_ratio', 'delta_theta_ratio', 'theta_alpha_ratio', 'delta_beta_ratio', 'low_high_ratio', 'line_length_channel_mean', 'line_length_channel_std', 'rms_channel_mean', 'rms_channel_std', 'std_channel_mean', 'std_channel_std', 'delta_power_channel_mean', 'delta_power_channel_std', 'theta_power_channel_mean', 'theta_power_channel_std', 'alpha_power_channel_mean', 'alpha_power_channel_std', 'beta_power_channel_mean', 'beta_power_channel_std', 'gamma_power_channel_mean', 'gamma_

## Step 5: Model Training

In [ ]:
param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [10, 20],
    "min_samples_leaf": [1, 5],
}

rf = RandomForestClassifier(
    class_weight="balanced_subsample", random_state=42, n_jobs=-1
)

grid = GridSearchCV(
    rf,
    param_grid,
    scoring="recall",  # prioritize seizure detection
    cv=3,
    n_jobs=10,
    verbose=2,
)

grid.fit(X_train, y_train)

rf_model = grid.best_estimator_

print("Best params:", grid.best_params_)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best params: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 200}


## Step 6: Model Evaluation

In [11]:
# Random Forest
print("\n Random Forest")

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf))

cm_rf = confusion_matrix(y_test, y_pred_rf)
print("\nConfusion Matrix:")
print(cm_rf)

tn, fp, fn, tp = cm_rf.ravel()
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print("Seizure Recall:", recall)

print("\nPerformance summary:")
print(classification_report(y_test, y_pred_rf, zero_division=0))


 Random Forest
Accuracy: 0.9044477129598939
ROC AUC: 0.5605409662776961

Confusion Matrix:
[[87311  9183]
 [   42     8]]
TN: 87311, FP: 9183, FN: 42, TP: 8
Seizure Recall: 0.16

Performance summary:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95     96494
           1       0.00      0.16      0.00        50

    accuracy                           0.90     96544
   macro avg       0.50      0.53      0.48     96544
weighted avg       1.00      0.90      0.95     96544



## Step 7: Threshold Tuning

In [ ]:
print("\nAutomatic Threshold Search")

thresholds = np.linspace(0.01, 0.99, 100)
rows = []

for t in thresholds:
    # Checks if probability is greater then current threshold
    y_pred_t = (y_proba_rf >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    rows.append(
        {
            "threshold": t,
            "accuracy": accuracy_score(y_test, y_pred_t),
            "precision": precision_score(y_test, y_pred_t, zero_division=0),
            "recall": recall_score(y_test, y_pred_t, zero_division=0),
            "f1": f1_score(y_test, y_pred_t, zero_division=0),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

threshold_df = pd.DataFrame(rows)

# Print 10 best
display(threshold_df.sort_values("recall", ascending=False).head(10))


# Choose best threshold
target_recall = 0.50

candidates = threshold_df[threshold_df["recall"] >= target_recall]

if len(candidates) > 0:
    # Best row is the one with lowest fp, from dandidates
    best_row = candidates.sort_values("FP").iloc[0]
else:
    # If candidates empty we go with highest recall
    best_row = threshold_df.sort_values("recall", ascending=False).iloc[0]

best_threshold = best_row["threshold"]

print("\nSelected Threshold")
print(best_row)

# Final evaluation at best threshold
print("\nFinal Evaluation (Best Threshold)")

y_pred_best = (y_proba_rf >= best_threshold).astype(int)

print("Threshold:", best_threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("ROC AUC:", roc_auc_score(y_test, y_proba_rf))

cm_best = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:")
print(cm_best)

tn, fp, fn, tp = cm_best.ravel()
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
print(f"TN: {tn}, FP: {fp}, FN: {fn}, TP: {tp}")
print("Seizure Recall:", recall)

print("\nPerformance summary:")
print(classification_report(y_test, y_pred_best, zero_division=0))


Automatic Threshold Search


,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
0,0.010000,0.009457,0.000523,1.00,0.001045,863,95631,0,50
1,0.019899,0.048879,0.000533,0.98,0.001066,4670,91824,1,49
2,0.029798,0.094206,0.000560,0.98,0.001119,9046,87448,1,49
3,0.039697,0.143240,0.000520,0.86,0.001039,13786,82708,7,43
4,0.049596,0.191395,0.000525,0.82,0.001049,18437,78057,9,41
5,0.059495,0.230858,0.000525,0.78,0.001049,22249,74245,11,39
6,0.069394,0.265910,0.000536,0.76,0.001071,25634,70860,12,38
7,0.079293,0.296414,0.000515,0.70,0.001029,28582,67912,15,35
8,0.089192,0.323656,0.000536,0.70,0.001071,31212,65282,15,35
9,0.099091,0.349385,0.000541,0.68,0.001081,33697,62797,16,34



Selected Threshold
threshold        0.237677
accuracy         0.642546
precision        0.000724
recall           0.500000
f1               0.001447
TN           62009.000000
FP           34485.000000
FN              25.000000
TP              25.000000
Name: 23, dtype: float64

Final Evaluation (Best Threshold)
Threshold: 0.23767676767676768
Accuracy: 0.642546403712297
ROC AUC: 0.5605409662776961

Confusion Matrix:
[[62009 34485]
 [   25    25]]
TN: 62009, FP: 34485, FN: 25, TP: 25
Seizure Recall: 0.5

Performance summary:
              precision    recall  f1-score   support

           0       1.00      0.64      0.78     96494
           1       0.00      0.50      0.00        50

    accuracy                           0.64     96544
   macro avg       0.50      0.57      0.39     96544
weighted avg       1.00      0.64      0.78     96544

